## Imports

In [ ]:
import cv2
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, WeightedRandomSampler, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from shared.utils import robust_minmax
from shared.early_stopping import EarlyStopping
from shared.mat_reader import MatReader
from shared.constants import CLASS_NAMES

device = "mps"

## Dataset

In [ ]:
Datapoint = tuple[torch.Tensor, int, str]  # (patch tensor, class label, patient id)

class ElasticDataset(Dataset[Datapoint]):
    def __init__(
        self,
        mat_reader: MatReader,
        eff_fov_indices: list[int],
        train: bool = False,
    ) -> None:

        self.mat_reader = mat_reader
        self.eff_fov_indices = eff_fov_indices
        self.train = train

        self.transform = A.Compose([
            A.ElasticTransform(
                alpha=10,      
                sigma=6, 
                # alpha_affine=100 * 0.03, 
                border_mode=cv2.BORDER_REFLECT_101,
                p=1.0,
            ),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            ToTensorV2() # Converts back to (C, H, W) and makes it a Tensor
        ])

    def __len__(self) -> int:
        return len(self.eff_fov_indices)

    def __getitem__(self, idx: int) -> Datapoint:
        eff_idx = self.eff_fov_indices[idx]

        image = self.mat_reader.images[eff_idx] # (C, H, W)
        image = robust_minmax(image).astype(np.float32)
        
        if self.train:
            image = image.transpose(1, 2, 0) # (H, W, C) for albumentations
            image = self.transform(image=image)['image']
        else:
            image = torch.from_numpy(image).float()

        class_label = int(self.mat_reader.class_labels[eff_idx])
        patient_id = str(self.mat_reader.patient_ids[eff_idx])

        return image, class_label, patient_id


## Model

In [ ]:
import pretrained_microscopy_models as pmm
import torch.utils.model_zoo as model_zoo
from torchvision import models

class CustomModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        # weights = models.ResNet18_Weights.DEFAULT
        # model = models.resnet18(weights=weights)
        
        model = models.resnet50(weights=None)
        url = pmm.util.get_pretrained_microscopynet_url("resnet50", "micronet")
        model.load_state_dict(model_zoo.load_url(url, map_location=device))
        
        in_features = model.fc.in_features
        num_classes = len(CLASS_NAMES)
        # model.fc = nn.Sequential(  # type: ignore[assignment]
        #     nn.Linear(in_features, num_classes),
        #     nn.Dropout(p=0.5),
        # )
        model.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(512, num_classes) 
        )

        for param in model.parameters():
            param.requires_grad = False
        for param in model.fc.parameters():
            param.requires_grad = True
        
        self.model = model
        self.fc = model.fc

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

## StratifiedGroupKFolds

In [ ]:
def unique_patients():
    patients = {}
    min_patients = 9999
    for i, class_name in enumerate(CLASS_NAMES):
        class_indices = np.where(mat_reader.class_labels == i)
        patient_ids = mat_reader.patient_ids[class_indices]
        unique_patient_ids, count = np.unique(patient_ids, return_counts=True)
        patients[class_name] = unique_patient_ids
        min_patients = min(min_patients, unique_patient_ids.shape[0])
    return patients, min_patients

mat_reader = MatReader("/Users/james/GitHub/lampe/lampe_dataset/3x3 bad SHG removed/")
n_splits = min(unique_patients()[1], 10)
print(f"# splits: {n_splits}")

from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True)
# sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

X, y, groups = mat_reader.images, mat_reader.class_labels, mat_reader.patient_ids

train_losses = [] # per fold loss curve
train_targets = [] # per fold confusion matrix
train_preds = [] # per fold confusion matrix

val_losses = []
val_targets = []
val_preds = []

accuracies = [] # per fold. can be averaged to get overall

# overall, taken from best epoch in each fold
best_val_targets = []
best_val_preds = []

for fold, (train_indices, val_indices) in enumerate(sgkf.split(X, y, groups=groups)):
    train_dataset = ElasticDataset(mat_reader, eff_fov_indices=train_indices.tolist(), train=True)
    val_dataset = ElasticDataset(mat_reader, eff_fov_indices=val_indices.tolist(), train=False)

    train_losses.append([])
    train_targets.append([])
    train_preds.append([])
    
    val_losses.append([])
    val_targets.append([])
    val_preds.append([])

    accuracies.append(0)

    candidate_best_val_targets = []
    candidate_best_val_preds = []

    # assumes class label ordering
    class_weights = 1.0 / np.bincount(mat_reader.class_labels[train_indices])
    sample_weights = class_weights[mat_reader.class_labels[train_indices]]
    
    sampler = WeightedRandomSampler(
        weights=sample_weights, 
        num_samples=len(sample_weights), 
        replacement=True
    )

    train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    model = CustomModel().to(device)

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=0.0001, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6
    )
    criterion = nn.CrossEntropyLoss()
    early_stopping = EarlyStopping(patience=10)

    num_epochs = 50

    for epoch in range(num_epochs):
        model.eval()
        model.fc.train()
        running_train_loss = 0.0
        for batch in train_loader:
            images, class_labels, patient_ids = batch
            # print(np.unique(class_labels, return_counts=True))
    
            inputs = images.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)

            targets = class_labels.to(device)
            loss = criterion(outputs, targets)

            # aggregate batch at fold level
            train_targets[fold].extend(list(targets.cpu().numpy()))
            train_preds[fold].extend(list(preds.cpu().numpy()))

            running_train_loss += loss.item() * images.size(0)
            
            loss.backward()
            optimizer.step()
    
        train_loss = running_train_loss / len(train_dataset)
        train_losses[fold].append(train_loss)

        with torch.no_grad():
            model.eval()
            running_val_loss = 0.0
            for batch in val_loader:
                images, class_labels, patient_ids = batch
                
                inputs = images.to(device)
                outputs = model(inputs)
                preds = torch.argmax(outputs, dim=1)
    
                targets = class_labels.to(device)
                loss = criterion(outputs, targets)

                # aggregate batch at fold level
                candidate_best_val_targets = targets.cpu().numpy()
                candidate_best_val_preds = preds.cpu().numpy()
                val_targets[fold].extend(list(candidate_best_val_targets))
                val_preds[fold].extend(list(candidate_best_val_preds))
                
                running_val_loss += loss.item() * images.size(0)
                
            val_loss = running_val_loss / len(val_dataset)
            val_losses[fold].append(val_loss)

        # take accuracy from last epoch in the fold
        acc = float(accuracy_score(val_targets[fold], val_preds[fold]))
        accuracies[fold] = acc

        # take predictions from last epoch in the fold (for Precision, Recall, F1)
        best_val_targets.extend(candidate_best_val_targets)
        best_val_preds.extend(candidate_best_val_preds)
        
        scheduler.step(val_loss)
        early_stopping(val_loss)
        if early_stopping.early_stop:
            print(f"Early stopping triggered at epoch {epoch + 1}")
            break

        print(
            (
                f"- Fold {fold + 1}/{n_splits}"
                f"- Epoch {epoch + 1}/{num_epochs}"
                f"- Train Loss: {train_loss:.4f}"
                f"- Val Loss: {val_loss:.4f}"
                f"- Accuracy: {acc:.4f}"
            )
        )

        

## Loss Curves

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots(1, n_splits, figsize=(5 * n_splits, 5))

# fig_title = "Report"
# fig.suptitle(fig_title)

# ax[0].imshow(img)
# ax[0].set_title('Original RGB')
# ax[0].axis('off')

for i in range(n_splits):
    ax[i].plot(train_losses[i], label="Train Loss")
    ax[i].plot(val_losses[i], label="Val Loss")
    ax[i].set_xlabel("Epoch")
    ax[i].set_ylabel("Loss")
    ax[i].set_title(f"Fold {i + 1}")
    ax[i].legend()


## Performance Metrics

In [ ]:
for fold in range(n_splits):
    cm = confusion_matrix(train_targets[fold], train_preds[fold])
    print(f"Train Confusion Matrix (Fold {fold+1}):")
    header = "          " + "  ".join(f"{name:>10}" for name in CLASS_NAMES)
    print(header)
    for i, row in enumerate(cm):
        row_str = "  ".join(f"{v:>10}" for v in row)
        print(f"{CLASS_NAMES[i]:>10}  {row_str}")

In [ ]:
for fold in range(n_splits):
    cm = confusion_matrix(val_targets[fold], val_preds[fold])
    print(f"Validation Confusion Matrix (Fold {fold+1} - {accuracies[fold] * 100:.2f}%):")
    header = "          " + "  ".join(f"{name:>10}" for name in CLASS_NAMES)
    print(header)
    for i, row in enumerate(cm):
        row_str = "  ".join(f"{v:>10}" for v in row)
        print(f"{CLASS_NAMES[i]:>10}  {row_str}")


## Report

In [ ]:
print("\n--- Final Results ---")
print(
    f"Mean CV Accuracy: {np.mean(accuracies) * 100:.2f}% ± {np.std(accuracies) * 100:.2f}%"
)

print("\nGlobal Classification Report:")
print(classification_report(best_val_targets, best_val_preds, target_names=CLASS_NAMES)) # must take only from last epochs
